In [1]:
%cd /content
!rm -rf ugc-admission-explainer
!git clone https://github.com/Ayesha200352/ugc-admission-explainer.git
%cd ugc-admission-explainer
!git checkout feature/rag-pipeline
!pwd

/content
Cloning into 'ugc-admission-explainer'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 42 (delta 13), reused 34 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 18.75 MiB | 44.14 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/ugc-admission-explainer
Branch 'feature/rag-pipeline' set up to track remote branch 'feature/rag-pipeline' from 'origin'.
Switched to a new branch 'feature/rag-pipeline'
/content/ugc-admission-explainer


In [2]:
!pip -q install langchain langchain-community langchain-text-splitters
!pip -q install chromadb sentence-transformers pypdf langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [3]:
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [4]:
from langchain_community.document_loaders import PyPDFLoader
import os

data_folders = [
    "data/faculty_sections",
    "data/zscore_reports",
    "data/policy_notices",
]

all_docs = []
for folder in data_folders:
    for filename in os.listdir(folder):
        if filename.endswith(".pdf"):
            path = os.path.join(folder, filename)
            loader = PyPDFLoader(path)
            docs = loader.load()
            all_docs.extend(docs)
            print(f"Loaded {len(docs)} pages from {filename}")

print(f"\nTotal pages loaded: {len(all_docs)}")

/tmp/ipykernel_1877/3361330323.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 6 pages from management.pdf
Loaded 18 pages from bio_science.pdf
Loaded 12 pages from arts_stream.pdf
Loaded 9 pages from engineering.pdf
Loaded 10 pages from zscore_cutoffs_2024_2025_handbook.pdf
Loaded 6 pages from 2040_Special_Intake_Cut off 2024_2025.pdf
Loaded 11 pages from zscore_cutoffs_2025_2026_handbook.pdf
Loaded 8 pages from section1_admissions_policy.pdf
Loaded 1 pages from Admissions Policy.pdf

Total pages loaded: 81


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(all_docs)

print("Total Chunks:", len(chunks))

Total Chunks: 1271


In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="chroma_db"
)

print(f"Stored {vectordb._collection.count()} chunks in Chroma")

/tmp/ipykernel_1877/1705869491.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Stored 1271 chunks in Chroma


In [7]:
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

test_queries = [
    "What Z-score was required for Engineering in the Colombo district in 2024?",
    "What subject combination is required for a Bio Science degree?",
    "How does the district quota system work?",
    "What is the minimum mark required in the Common General Paper?",
    "Which universities offer Engineering degrees?",
]

for q in test_queries:
    print(f"\n=== QUERY: {q} ===")
    results = retriever.invoke(q)
    for i, r in enumerate(results):
        print(f"--- Result {i+1} (source: {r.metadata.get('source', 'unknown')}) ---")
        print(r.page_content[:300])
        print()


=== QUERY: What Z-score was required for Engineering in the Colombo district in 2024? ===
--- Result 1 (source: data/zscore_reports/zscore_cutoffs_2025_2026_handbook.pdf) ---
even though they have obtained the minimum Z-score required to get selected.
COLOMBO
GAMPAHA
KALUTARA
MATALE
KANDY
NUWARA ELIYA
GALLE
MATARA
HAMBANTOTA
JAFFNA
KILINOCHCHI
MANNAR
MULLAITIVU
VAVUNIYA
TRINCOMALEE
BATTICALOA
AMPARA
PUTTALAM
KURUNEGALA
ANURADHAPURA
POLONNARUWA
BADULLA
MONARAGALA
KEGALLE

--- Result 2 (source: data/zscore_reports/zscore_cutoffs_2025_2026_handbook.pdf) ---
even though they have obtained the minimum Z-score required to get selected.
COLOMBO
GAMPAHA
KALUTARA
MATALE
KANDY
NUWARA ELIYA
GALLE
MATARA
HAMBANTOTA
JAFFNA
KILINOCHCHI
MANNAR
MULLAITIVU
VAVUNIYA
TRINCOMALEE
BATTICALOA
AMPARA
PUTTALAM
KURUNEGALA
ANURADHAPURA
POLONNARUWA
BADULLA
MONARAGALA
KEGALLE

--- Result 3 (source: data/zscore_reports/zscore_cutoffs_2025_2026_handbook.pdf) ---
even though they have obtained the minimum Z-score r

In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

response = llm.invoke("What is a district quota system?")
print(response.content)

A district quota system is a method used in various contexts, such as politics, education, or employment, to allocate a certain number of positions or opportunities to a specific group or region. The system is based on a quota, which is a fixed number or percentage of the total available positions that are reserved for the designated group or region.

In a district quota system, the total number of positions is divided into a certain number of quotas, each representing a specific district or region. The number of quotas allocated to each district is usually based on the population size, electoral strength, or other relevant factors.

Here's a simplified example of how a district quota system might work:

Let's say there are 10 positions available and 5 districts. The quota system allocates 2 positions to each district, based on the population size or other factors. The districts are then ranked or selected in a predetermined order, and the positions are filled according to the quota.

